# AutoSolve Standalone & Independent ML Training Pipeline

This notebook is a **completely self-contained, independent training suite** for the AutoSolve Blender camera tracking addon. It contains all schemas, feature extraction, PyTorch training loops, and ONNX/NumPy model exporters directly inside the cells below. No external Python files are required to run this notebook.

### 🔄 Workflow Overview:
1. **Environment Setup**: Mount Google Drive and install requirements (`torch`, `onnx`, `onnxruntime`, `opencv-python`).
2. **Ingestion & Feature Extraction**: Extract optical flow, noise, and track trajectories directly from reference video clips.
3. **Settings Expected-Reward Model**: Train a 28-dimensional MLP to predict solve quality and rank presets.
4. **Track Quality Predictor**: Train a 15-dimensional MLP to proactively retire unstable tracking markers.
5. **Presets Optimizer & Export**: Perform grid search recommendation and save `.onnx` and JSON models.

## 📂 Google Drive Persistent Storage (Recommended for Colab)
Link directories directly to your Google Drive to persist video clips and trained checkpoints.

In [ ]:
#@title Configure Google Drive Integration
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_PROJECT_PATH = "AutoSolve_ML_Data" #@param {type:"string"}

import os
import shutil

in_colab = False
try:
    import google.colab
    in_colab = True
except ImportError:
    pass

if in_colab and USE_GOOGLE_DRIVE:
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    persist_dir = f"/content/drive/MyDrive/{DRIVE_PROJECT_PATH}"
    os.makedirs(persist_dir, exist_ok=True)
    
    persistent_clips_dir = os.path.join(persist_dir, "clips")
    persistent_data_dir = os.path.join(persist_dir, "data")
    persistent_runs_dir = os.path.join(persist_dir, "runs")
    
    for d in [persistent_clips_dir, persistent_data_dir, persistent_runs_dir]:
        os.makedirs(d, exist_ok=True)
        
    print(f"\n📂 Google Drive paths mapped:")
    print(f"   Clips folder:    {persistent_clips_dir}")
    print(f"   Datasets folder: {persistent_data_dir}")
    print(f"   Runs folder:     {persistent_runs_dir}")
    
    # Link directories
    for path, target in [("ml/clips", persistent_clips_dir), ("ml/data", persistent_data_dir), ("ml/runs", persistent_runs_dir)]:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if os.path.exists(path):
            if os.path.islink(path): os.unlink(path)
            elif os.path.isdir(path): shutil.rmtree(path)
            else: os.remove(path)
        os.symlink(target, path)
        print(f"✅ Linked '{path}' -> '{target}'")
else:
    print("Using temporary local filesystem inside notebook environment.")
    for d in ['ml/clips', 'ml/data', 'ml/runs', 'ml/data/raw', 'ml/data/processed', 'ml/data/live']:
        os.makedirs(d, exist_ok=True)
    print("👉 Upload your video clips directly to 'ml/clips/' and solve logs to 'ml/data/raw/'.")

In [ ]:
# Install dependencies
!pip install -q torch numpy onnx onnxruntime opencv-python

import torch, numpy as np, onnx, onnxruntime as ort
print(f"PyTorch:     {torch.__version__}")
print(f"NumPy:       {np.__version__}")
print(f"ONNX:        {onnx.__version__}")
print(f"onnxruntime: {ort.__version__}")
print(f"GPU (CUDA):  {torch.cuda.is_available()}")

## 🛠️ Step 1: Standalone Schemas and Metadata Definitions

In [ ]:
from dataclasses import dataclass, asdict
from typing import List, Tuple, Dict, Any

@dataclass
class ClipMetadata:
    clip_name: str
    width: int
    height: int
    fps: float
    frame_count: int

@dataclass
class TrackingSettings:
    quality_preset: str
    footage_type: str
    robust_mode: bool
    tripod_mode: bool
    pattern_size: int
    search_size: int
    correlation: float
    threshold: float
    motion_model: str

@dataclass
class TrackSample:
    track_name: str
    region: str
    positions: List[Tuple[float, float]]
    velocities: List[Tuple[float, float]]
    jitter_scores: List[float]
    lifespan: int
    survived: bool
    has_bundle: bool
    average_error: float

@dataclass
class SolveSample:
    clip_metadata: ClipMetadata
    settings: TrackingSettings
    tracks: List[TrackSample]
    solve_success: bool
    solve_error: float
    bundle_count: int
    bundle_ratio: float
    runtime_seconds: float

def solve_sample_to_dict(sample: SolveSample) -> Dict[str, Any]:
    return asdict(sample)

## 🎥 Step 2: Feature Ingestion & Dataset Preparation

In [ ]:
import cv2

def extract_features_from_video(video_path: str) -> dict:
    """Extract average motion, zoom, distortion, and noise metrics directly from clip."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video file: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    step = max(1, frame_count // 30)
    prev_gray = None
    motions, divergences, noises, curvatures = [], [], [], []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret: break

        if frame_idx % step == 0:
            small_frame = cv2.resize(frame, (320, 180))
            gray = cv2.cvtColor(small_frame, cv2.COLOR_BGR2GRAY)

            # Grain Noise
            blurred = cv2.GaussianBlur(gray, (5, 5), 0)
            high_freq = cv2.absdiff(gray, blurred)
            noises.append(float(np.mean(high_freq)))

            # Line Curvature
            edges = cv2.Canny(gray, 50, 150)
            lines = cv2.HoughLinesP(edges, 1, np.pi/180, 50, minLineLength=30, maxLineGap=10)
            if lines is not None:
                angles = [np.arctan2(l[0][3] - l[0][1], l[0][2] - l[0][0]) * 180 / np.pi for l in lines]
                curvatures.append(float(np.var(angles)) if angles else 0.0)
            else:
                curvatures.append(0.0)

            # Dense Flow
            if prev_gray is not None:
                flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                fx, fy = flow[..., 0], flow[..., 1]
                motions.append(float(np.mean(np.sqrt(fx**2 + fy**2))))
                div = np.gradient(fx, axis=1) + np.gradient(fy, axis=0)
                divergences.append(float(np.mean(div)))
            prev_gray = gray
        frame_idx += 1

    cap.release()
    return {
        "clip_name": os.path.splitext(os.path.basename(video_path))[0],
        "width": width, "height": height, "fps": fps, "frame_count": frame_count,
        "mean_motion": float(np.mean(motions)) if motions else 0.0,
        "zoom_divergence": float(np.mean(np.abs(divergences))) if divergences else 0.0,
        "distortion_factor": float(np.mean(curvatures)) if curvatures else 0.0,
        "noise_ratio": float(np.mean(noises)) if noises else 0.0
    }

In [ ]:
import math
import random

def classify_region(fx: float, fy: float) -> str:
    ry = "top" if fy > 0.66 else ("bottom" if fy < 0.33 else "mid")
    rx = "left" if fx < 0.33 else ("right" if fx > 0.66 else "center")
    return "center" if (ry == "mid" and rx == "center") else (f"{ry}-{rx}" if ry != "mid" else f"mid-{rx}")

def run_opencv_tracking(video_path: str, grid_size: int = 8) -> Tuple[List[List[Tuple[float, float]]], dict]:
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    ret, frame = cap.read()
    gray_prev = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    pts = cv2.goodFeaturesToTrack(gray_prev, maxCorners=grid_size*grid_size, qualityLevel=0.01, minDistance=20)
    if pts is None:
        pts = np.array([[[x, y]] for y in np.linspace(height*0.1, height*0.9, grid_size) for x in np.linspace(width*0.1, width*0.9, grid_size)], dtype=np.float32)

    num_tracks = len(pts)
    trajectories = [[(float(p[0][0]/width), float(p[0][1]/height))] for p in pts]
    active = np.ones(num_tracks, dtype=bool)
    p_prev = pts.copy()

    while True:
        ret, frame = cap.read()
        if not ret: break
        gray_curr = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        p_curr, status, _ = cv2.calcOpticalFlowPyrLK(gray_prev, gray_curr, p_prev, None, winSize=(21, 21), maxLevel=3)

        for idx in range(num_tracks):
            if not active[idx]: continue
            if status[idx][0] == 0:
                active[idx] = False
                continue
            x, y = p_curr[idx][0]
            if x < 0 or x >= width or y < 0 or y >= height:
                active[idx] = False
                continue
            trajectories[idx].append((float(x/width), float(y/height)))
        p_prev = p_curr
        gray_prev = gray_curr
    cap.release()
    return trajectories, {"clip_name": os.path.splitext(os.path.basename(video_path))[0], "width": width, "height": height, "fps": fps, "frame_count": frame_count}

In [ ]:
def simulate_solve_attempts(clips_dir="ml/clips", out_dir="ml/data/raw"):
    os.makedirs(out_dir, exist_ok=True)
    supported = {'.mp4', '.mov', '.avi', '.mkv', '.ogg', '.webm'}
    files = [os.path.join(clips_dir, f) for f in os.listdir(clips_dir) if os.path.splitext(f)[1].lower() in supported] if os.path.exists(clips_dir) else []
    
    if not files:
        print("No clips found in 'ml/clips/'. Generating synthetic trajectories for training...")
        # Generate dummy dataset if no video files are loaded
        for i in range(5):
            clip_name = f"dummy_clip_{i}"
            # Create dummy features metadata
            with open(os.path.join(out_dir, f"{clip_name}_video_meta.json"), 'w') as fh:
                json.dump({"clip_name": clip_name, "width": 1920, "height": 1080, "fps": 30.0, "frame_count": 250, "mean_motion": 0.6, "zoom_divergence": 0.0, "distortion_factor": 0.0, "noise_ratio": 0.003}, fh, indent=4)
            # Create solve variation attempt
            solve = {
                "clip_metadata": {"clip_name": clip_name, "width": 1920, "height": 1080, "fps": 30.0, "frame_count": 250},
                "settings": {"quality_preset": "BALANCED", "footage_type": "AUTO", "robust_mode": False, "tripod_mode": False, "pattern_size": 17, "search_size": 71, "correlation": 0.7, "threshold": 0.3, "motion_model": "LocRot"},
                "tracks": [
                    {"track_name": f"t_{j}", "region": "center", "positions": [(0.5, 0.5)]*100, "velocities": [(0.0, 0.0)]*99, "jitter_scores": [0.0]*98, "lifespan": 100, "survived": True, "has_bundle": True, "average_error": 0.2}
                    for j in range(40)
                ],
                "solve_success": True, "solve_error": 0.3, "bundle_count": 35, "bundle_ratio": 0.875, "runtime_seconds": 5.0
            }
            with open(os.path.join(out_dir, f"{clip_name}_balanced_standard.json"), 'w') as fh:
                json.dump(solve, fh, indent=4)
        return
        
    print(f"Processing {len(files)} clips...")
    for fp in files:
        clip_name = os.path.splitext(os.path.basename(fp))[0]
        try:
            # Save features
            v_feats = extract_features_from_video(fp)
            with open(os.path.join(out_dir, f"{clip_name}_video_meta.json"), 'w', encoding='utf-8') as fh:
                json.dump(v_feats, fh, indent=4)
            
            # Extract trajectories & simulate settings variations
            trajectories, meta = run_opencv_tracking(fp)
            variations = [
                ("BALANCED", False, False, "balanced_standard"),
                ("FAST", False, False, "fast_standard"),
                ("QUALITY", False, False, "quality_standard"),
                ("BALANCED", True, False, "balanced_robust"),
                ("BALANCED", False, True, "balanced_tripod")
            ]
            for q, r, t, suffix in variations:
                solve = simulate_variation(trajectories, meta, q, r, t)
                with open(os.path.join(out_dir, f"{clip_name}_{suffix}.json"), 'w') as fh:
                    json.dump(solve, fh, indent=4)
        except Exception as e:
            print(f"  Failed {clip_name}: {e}")

In [ ]:
FOOTAGE_TYPES = ['AUTO', 'INDOOR', 'OUTDOOR', 'DRONE', 'HANDHELD', 'GIMBAL', 'ACTION', 'VFX', 'SCREEN', 'CINEMATIC']
MOTION_MODELS = ['Loc', 'LocRot', 'Affine', 'Perspective']

def get_one_hot(value: str, catalog: List[str]) -> List[float]:
    one_hot = [0.0] * len(catalog)
    if value in catalog: one_hot[catalog.index(value)] = 1.0
    else: one_hot[0] = 1.0
    return one_hot

def extract_sample_features(sample: dict, video_meta: dict = None) -> List[float]:
    meta = sample["clip_metadata"]
    settings = sample["settings"]
    clip_feats = [float(meta.get("width", 1920)), float(meta.get("height", 1080)), float(meta.get("fps", 24.0)), float(meta.get("frame_count", 250))]
    switches = [1.0 if settings.get("tripod_mode", False) else 0.0, 1.0 if settings.get("robust_mode", False) else 0.0]
    setting_feats = [float(settings.get("pattern_size", 17)), float(settings.get("search_size", 71)), float(settings.get("correlation", 0.70)), float(settings.get("threshold", 0.30))]
    f_type_oh = get_one_hot(settings.get("footage_type", "AUTO"), FOOTAGE_TYPES)
    m_model_oh = get_one_hot(settings.get("motion_model", "LocRot"), MOTION_MODELS)
    if video_meta is None: video_meta = {}
    v_feats = [float(video_meta.get("mean_motion", 0.5)), float(video_meta.get("zoom_divergence", 0.0)), float(video_meta.get("distortion_factor", 0.0)), float(video_meta.get("noise_ratio", 0.003))]
    return clip_feats + switches + setting_feats + f_type_oh + m_model_oh + v_feats

def calculate_reward(sample: dict) -> float:
    success = 1.0 if sample.get("solve_success", False) else 0.0
    error = sample.get("solve_error", 10.0)
    bundle_ratio = sample.get("bundle_ratio", 0.0)
    error_penalty = max(0.0, min(1.0, error / 5.0))
    return round(success * (1.0 - error_penalty) * bundle_ratio, 4)

def prepare_dataset(data_dir="ml/data/raw", output_json="ml/data/processed/settings_dataset.json"):
    video_meta_lookup = {}
    real_samples = []
    if os.path.exists(data_dir):
        for f in os.listdir(data_dir):
            fp = os.path.join(data_dir, f)
            if f.endswith('_video_meta.json'):
                try:
                    with open(fp) as fh: m = json.load(fh); video_meta_lookup[m["clip_name"]] = m
                except: pass
            elif f.endswith('.json') and not f.endswith('settings_dataset.json'):
                try:
                    with open(fp) as fh: s = json.load(fh); real_samples.append(s)
                except: pass
                
    by_clip = {}
    for s in real_samples:
        by_clip.setdefault(s["clip_metadata"]["clip_name"], []).append(s)
    clip_names = list(by_clip.keys())
    random.shuffle(clip_names)
    split_idx = int(len(clip_names) * 0.8)
    train_clips = set(clip_names[:split_idx])
    train_raw, val_raw = [], []
    for c, samples in by_clip.items():
        if c in train_clips: train_raw.extend(samples)
        else: val_raw.extend(samples)
        
    train_X = [extract_sample_features(s, video_meta_lookup.get(s["clip_metadata"]["clip_name"])) for s in train_raw]
    train_y = [calculate_reward(s) for s in train_raw]
    val_X = [extract_sample_features(s, video_meta_lookup.get(s["clip_metadata"]["clip_name"])) for s in val_raw]
    val_y = [calculate_reward(s) for s in val_raw]
    
    num_features = len(train_X[0])
    means, stds = [0.0]*num_features, [1.0]*num_features
    for j in range(num_features):
        col = [train_X[i][j] for i in range(len(train_X))]
        means[j] = sum(col) / len(col)
        variance = sum((x - means[j])**2 for x in col) / len(col)
        stds[j] = math.sqrt(variance) if variance > 1e-8 else 1.0
        
    def normalize(X):
        return [[(val - m)/s for val, m, s in zip(row, means, stds)] for row in X]
        
    dataset = {
        "train": {"X": normalize(train_X), "y": train_y},
        "val": {"X": normalize(val_X), "y": val_y},
        "input_mean": means, "input_std": stds
    }
    os.makedirs(os.path.dirname(output_json), exist_ok=True)
    with open(output_json, 'w') as fh:
        json.dump(dataset, fh, indent=4)
    print(f"Ingestion completed. Dataset saved to {output_json}")

In [ ]:
# Execute Ingestion pipeline
simulate_solve_attempts()
prepare_dataset()

## 🧠 Step 3: Train Expected Reward Optimizer MLP (28 Features)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

class SettingsMLP(nn.Module):
    """Expected reward MLP mapping 28 features to 1 expected reward score."""
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(28, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.network(x)

def train_settings_optimizer(epochs=50):
    with open("ml/data/processed/settings_dataset.json") as f:
        dataset = json.load(f)
    tx = torch.tensor(dataset["train"]["X"], dtype=torch.float32)
    ty = torch.tensor(dataset["train"]["y"], dtype=torch.float32).unsqueeze(1)
    vx = torch.tensor(dataset["val"]["X"], dtype=torch.float32)
    vy = torch.tensor(dataset["val"]["y"], dtype=torch.float32).unsqueeze(1)
    
    loader = DataLoader(TensorDataset(tx, ty), batch_size=16, shuffle=True)
    model = SettingsMLP()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    
    best_val_loss = float('inf')
    best_weights = None
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for bx, by in loader:
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()*bx.size(0)
        train_loss /= len(tx)
        
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(vx), vy).item()
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.cpu().numpy().tolist() for k, v in model.state_dict().items()}
        if (epoch+1)%10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d} | Train MSE: {train_loss:.5f} | Val MSE: {val_loss:.5f}")
            
    meta = {"weights": best_weights, "input_mean": dataset["input_mean"], "input_std": dataset["input_std"], "best_val_loss": best_val_loss}
    os.makedirs("ml/runs/settings_optimizer", exist_ok=True)
    with open("ml/runs/settings_optimizer/model_meta_weights.json", 'w') as fh:
        json.dump(meta, fh, indent=4)
    print("Settings Model weights saved.")

train_settings_optimizer()

## 🎯 Step 4: Train Track Quality Predictor MLP (15 Features)

In [ ]:
class TrackMLP(nn.Module):
    """15 -> 64 -> 32 -> 1 MLP for survival prediction."""
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(15, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.network(x)

def train_track_predictor(epochs=100):
    # Gather data from raw solves
    data_dir = "ml/data/raw"
    json_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.json') and not f.endswith('_video_meta.json') and not f.endswith('settings_dataset.json')] if os.path.exists(data_dir) else []
    
    X_samples, y_samples = [], []
    FOOTAGE_MAP = {f: i for i, f in enumerate(FOOTAGE_TYPES)}
    REGIONS_LIST = ['top-left', 'top-center', 'top-right', 'mid-left', 'center', 'mid-right', 'bottom-left', 'bottom-center', 'bottom-right']
    
    for fp in json_files:
        try:
            with open(fp) as fh: d = json.load(fh)
            f_idx = FOOTAGE_MAP.get(d["settings"]["footage_type"], 0)
            robust = d["settings"]["robust_mode"]
            tracks = d["tracks"]
            t_coords = {t["track_name"]: t["positions"] for t in tracks}
            for t in tracks:
                coords = t["positions"]
                survived = t["survived"] and t["has_bundle"]
                r_idx = REGIONS_LIST.index(t["region"]) if t["region"] in REGIONS_LIST else 4
                if len(coords) < 6: continue
                for f_idx_check in range(5, len(coords), 10):
                    sub = coords[:f_idx_check+1]
                    neighbors = [tc[:f_idx_check+1] for name, tc in t_coords.items() if name != t["track_name"] and len(tc) > f_idx_check]
                    # Extracted features
                    feats = np.zeros(15, dtype=np.float32)
                    inspect = sub[-6:]
                    v_x = [inspect[i][0] - inspect[i-1][0] for i in range(1, len(inspect))]
                    v_y = [inspect[i][1] - inspect[i-1][1] for i in range(1, len(inspect))]
                    feats[0], feats[1] = np.mean(v_x), np.mean(v_y)
                    feats[2] = np.std(v_x) if len(v_x) > 1 else 0.0
                    feats[3] = np.std(v_y) if len(v_y) > 1 else 0.0
                    acc_x = [v_x[i] - v_x[i-1] for i in range(1, len(v_x))] if len(v_x) > 1 else [0.0]
                    acc_y = [v_y[i] - v_y[i-1] for i in range(1, len(v_y))] if len(v_y) > 1 else [0.0]
                    feats[4], feats[5] = np.mean(acc_x), np.mean(acc_y)
                    feats[6] = sum(1 for i in range(1, len(v_x)) if (v_x[i] > 0) != (v_x[i-1] > 0))
                    feats[7] = sum(1 for i in range(1, len(v_y)) if (v_y[i] > 0) != (v_y[i-1] > 0))
                    feats[8] = len(coords)
                    # Neighbor features
                    min_dist = 999.0
                    n_vels_x, n_vels_y = [], []
                    for n_coords in neighbors:
                        if len(n_coords) >= 1:
                            dist = ((coords[-1][0] - n_coords[-1][0])**2 + (coords[-1][1] - n_coords[-1][1])**2)**0.5
                            if dist < min_dist: min_dist = dist
                            if len(n_coords) >= 2:
                                n_vels_x.append(n_coords[-1][0] - n_coords[-2][0])
                                n_vels_y.append(n_coords[-1][1] - n_coords[-2][1])
                    feats[9] = min_dist if min_dist < 998.0 else 1.0
                    if n_vels_x:
                        feats[10] = v_x[-1] - np.mean(n_vels_x)
                        feats[11] = v_y[-1] - np.mean(n_vels_y)
                    feats[12], feats[13] = float(r_idx), float(f_idx)
                    feats[14] = 1.0 if robust else 0.0
                    
                    X_samples.append(feats)
                    y_samples.append(1.0 if survived else 0.0)
        except: pass
        
    if not X_samples:
        print("No track samples found. Injecting synthetic vectors for training...")
        X_samples = np.random.randn(400, 15).astype(np.float32).tolist()
        y_samples = [1.0 if random.random() > 0.4 else 0.0 for _ in range(400)]
        
    X_arr = np.array(X_samples, dtype=np.float32)
    y_arr = np.array(y_samples, dtype=np.float32)
    mean = np.mean(X_arr, axis=0)
    std = np.std(X_arr, axis=0)
    std[std < 1e-6] = 1.0
    X_norm = (X_arr - mean) / std
    
    split = int(len(X_norm)*0.8)
    tx, ty = torch.tensor(X_norm[:split]), torch.tensor(y_arr[:split]).unsqueeze(1)
    vx, vy = torch.tensor(X_norm[split:]), torch.tensor(y_arr[split:]).unsqueeze(1)
    
    loader = DataLoader(TensorDataset(tx, ty), batch_size=32, shuffle=True)
    model = TrackMLP()
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_loss = float('inf')
    best_weights = None
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for bx, by in loader:
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()*bx.size(0)
        train_loss /= len(tx)
        
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(vx), vy).item()
            acc = np.mean((model(vx).numpy() > 0.5) == vy.numpy())
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.cpu().numpy().tolist() for k, v in model.state_dict().items()}
        if (epoch+1)%20 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:03d} | Train BCELoss: {train_loss:.4f} | Val BCELoss: {val_loss:.4f} | Val Acc: {acc:.1%}")
            
    meta = {"weights": best_weights, "input_mean": mean.tolist(), "input_std": std.tolist()}
    os.makedirs("ml/runs/track_predictor", exist_ok=True)
    with open("ml/runs/track_predictor/model_meta_weights.json", 'w') as fh:
        json.dump(meta, fh, indent=4)
    print("Track Predictor saved successfully.")

train_track_predictor()

## 📈 Step 5: Empirical Trackability Heatmap / Region Weights

In [ ]:
def compile_region_weights(data_dir="ml/data/raw", output_json="ml/runs/region_weights.json"):
    regions = ['top-left', 'top-center', 'top-right', 'mid-left', 'center', 'mid-right', 'bottom-left', 'bottom-center', 'bottom-right']
    prebaked = {
        "AUTO": {r: 1.0 for r in regions},
        "INDOOR": {r: 1.0 for r in regions},
        "OUTDOOR": {"top-left": 0.40, "top-center": 0.15, "top-right": 0.40, "mid-left": 0.85, "center": 0.95, "mid-right": 0.85, "bottom-left": 1.00, "bottom-center": 1.00, "bottom-right": 1.00},
        "DRONE": {"top-left": 0.10, "top-center": 0.05, "top-right": 0.10, "mid-left": 0.70, "center": 0.90, "mid-right": 0.70, "bottom-left": 1.00, "bottom-center": 1.00, "bottom-right": 1.00}
    }
    # Scan solves
    json_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.json') and not f.endswith('_video_meta.json') and not f.endswith('settings_dataset.json')] if os.path.exists(data_dir) else []
    weights = {}
    if json_files:
        stats = {}
        for fp in json_files:
            try:
                with open(fp) as fh: d = json.load(fh)
                f_type = d["settings"]["footage_type"]
                stats.setdefault(f_type, {})
                for t in d["tracks"]:
                    region = t["region"]
                    r_stats = stats[f_type].setdefault(region, {"detected": 0, "survived": 0})
                    r_stats["detected"] += 1
                    if t["survived"] and t["has_bundle"]: r_stats["survived"] += 1
            except: pass
            
        for f_type in ["AUTO", "INDOOR", "OUTDOOR", "DRONE"]:
            weights[f_type] = {}
            f_stats = stats.get(f_type, {})
            fallback_weights = prebaked.get(f_type, prebaked["AUTO"])
            for region, r_weights in fallback_weights.items():
                if region in f_stats and f_stats[region]["detected"] > 10:
                    weights[f_type][region] = f_stats[region]["survived"] / f_stats[region]["detected"]
                else:
                    weights[f_type][region] = r_weights
            max_w = max(weights[f_type].values())
            if max_w > 0:
                for r in weights[f_type]: weights[f_type][r] = round(weights[f_type][r]/max_w, 3)
    else:
        weights = prebaked
        
    with open(output_json, 'w') as fh:
        json.dump(weights, fh, indent=4)
    print(f"Empirical weights generated under {output_json}")

compile_region_weights()

## 📤 Step 6: Standalone Exporters (ONNX, NumPy, & Defaults Grid Search)

In [ ]:
import torch.onnx

def export_onnx_binaries(out_dir="ml/runs/onnx"):
    os.makedirs(out_dir, exist_ok=True)
    
    # 1. Settings model (28 Features)
    with open("ml/runs/settings_optimizer/model_meta_weights.json") as fh: s_meta = json.load(fh)
    s_model = SettingsMLP()
    s_model.load_state_dict({k: torch.tensor(v) for k, v in s_meta["weights"].items()})
    s_model.eval()
    s_onnx = os.path.join(out_dir, "settings_model.onnx")
    torch.onnx.export(
        s_model, torch.zeros(1, 28), s_onnx,
        input_names=["clip_features"], output_names=["reward"], opset_version=17,
        dynamic_axes={"clip_features": {0: "batch"}, "reward": {0: "batch"}}
    )
    with open(os.path.join(out_dir, "settings_model_meta.json"), 'w') as fh:
        json.dump({"input_mean": s_meta["input_mean"], "input_std": s_meta["input_std"], "input_size": 28, "output_size": 1}, fh, indent=2)
        
    # 2. Track Predictor model (15 Features)
    with open("ml/runs/track_predictor/model_meta_weights.json") as fh: t_meta = json.load(fh)
    t_model = TrackMLP()
    t_model.load_state_dict({k: torch.tensor(v) for k, v in t_meta["weights"].items()})
    t_model.eval()
    t_onnx = os.path.join(out_dir, "track_predictor.onnx")
    torch.onnx.export(
        t_model, torch.zeros(1, 15), t_onnx,
        input_names=["track_features"], output_names=["survival_prob"], opset_version=17,
        dynamic_axes={"track_features": {0: "batch"}, "survival_prob": {0: "batch"}}
    )
    with open(os.path.join(out_dir, "track_predictor_meta.json"), 'w') as fh:
        json.dump({"input_mean": t_meta["input_mean"], "input_std": t_meta["input_std"], "input_size": 15, "output_size": 1}, fh, indent=2)
        
    # 3. NumPy JSON model
    numpy_model = {
        "layer1_weight": t_meta["weights"]["network.0.weight"],
        "layer1_bias": t_meta["weights"]["network.0.bias"],
        "layer2_weight": t_meta["weights"]["network.2.weight"],
        "layer2_bias": t_meta["weights"]["network.2.bias"],
        "layer3_weight": t_meta["weights"]["network.4.weight"],
        "layer3_bias": t_meta["weights"]["network.4.bias"],
        "input_mean": t_meta["input_mean"], "input_std": t_meta["input_std"], "activation": "relu"
    }
    with open("ml/runs/track_predictor.json", 'w') as fh: json.dump(numpy_model, fh, indent=4)
    print("ONNX and JSON models exported successfully!")

export_onnx_binaries()

In [ ]:
# grid search defaultsPresets generator (28 features)
def matmul(A, B_T):
    N, K, M = len(A), len(A[0]), len(B_T)
    out = [[0.0]*M for _ in range(N)]
    for i in range(N):
        for j in range(M):
            out[i][j] = sum(A[i][k] * B_T[j][k] for k in range(K))
    return out

def predict_rewards(X, weights):
    x1 = [[max(0.0, sum(row[k]*weights["network.0.weight"][j][k] for k in range(28)) + weights["network.0.bias"][j]) for j in range(64)] for row in X]
    x2 = [[max(0.0, sum(row[k]*weights["network.2.weight"][j][k] for k in range(64)) + weights["network.2.bias"][j]) for j in range(32)] for row in x1]
    x3 = [1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, sum(row[k]*weights["network.4.weight"][0][k] for k in range(32)) + weights["network.4.bias"][0])))) for row in x2]
    return x3

def run_defaults_optimizer(output_json="ml/runs/recommended_defaults.json"):
    with open("ml/runs/settings_optimizer/model_meta_weights.json") as f: meta = json.load(f)
    means, stds, weights = meta["input_mean"], meta["input_std"], meta["weights"]
    
    grid_patterns = [11, 15, 17, 21, 31, 55]
    grid_searches = [51, 71, 91, 121, 231]
    grid_correlations = [0.55, 0.65, 0.70, 0.75, 0.85]
    grid_thresholds = [0.1, 0.2, 0.3, 0.4]
    grid_models = ['Loc', 'LocRot', 'Affine', 'Perspective']
    
    candidates = []
    for p in grid_patterns:
        for s in grid_searches:
            for c in grid_correlations:
                for t in grid_thresholds:
                    for m in grid_models:
                        candidates.append({"pattern_size": p, "search_size": s, "correlation": c, "threshold": t, "motion_model": m})
                        
    classes = ['HD_24fps', 'HD_30fps', 'HD_60fps', '4K_24fps', '4K_30fps']
    res_map = {'HD_24fps':(1920.0,1080.0,24.0,250.0), 'HD_30fps':(1920.0,1080.0,30.0,250.0), 'HD_60fps':(1920.0,1080.0,60.0,250.0), '4K_24fps':(3840.0,2160.0,24.0,250.0), '4K_30fps':(3840.0,2160.0,30.0,250.0)}
    
    recommendations = {}
    for r_class in classes:
        w, h, fps, frame_count = res_map[r_class]
        recommendations[r_class] = {}
        for f_type in ["AUTO", "INDOOR", "OUTDOOR", "DRONE"]:
            batch = []
            for cand in candidates:
                clip_feats = [w, h, fps, frame_count]
                switches = [0.0, 0.0]
                settings_feats = [float(cand["pattern_size"]), float(cand["search_size"]), cand["correlation"], cand["threshold"]]
                f_type_oh = get_one_hot(f_type, FOOTAGE_TYPES)
                m_model_oh = get_one_hot(cand["motion_model"], MOTION_MODELS)
                # Video features matching actual runtime defaults
                v_feats = [0.5, 0.0, 0.0, 0.003]
                if f_type == "ACTION": v_feats[0] = 3.5
                elif f_type == "DRONE": v_feats[1] = 0.5
                
                raw = clip_feats + switches + settings_feats + f_type_oh + m_model_oh + v_feats
                batch.append([(raw[k] - means[k])/stds[k] for k in range(28)])
                
            rewards = predict_rewards(batch, weights)
            max_idx = rewards.index(max(rewards))
            best = candidates[max_idx].copy()
            best["expected_reward"] = round(rewards[max_idx], 4)
            recommendations[r_class][f_type] = best
            
    with open(output_json, 'w') as fh: json.dump(recommendations, fh, indent=4)
    print(f"Recommended defaults saved to {output_json}")

run_defaults_optimizer()

In [ ]:
# Outputs Verification
expected = [
    'ml/runs/onnx/track_predictor.onnx',
    'ml/runs/onnx/track_predictor_meta.json',
    'ml/runs/onnx/settings_model.onnx',
    'ml/runs/onnx/settings_model_meta.json',
    'ml/runs/track_predictor.json',
    'ml/runs/region_weights.json',
    'ml/runs/recommended_defaults.json',
]
print("Trained Model Output Verification:\n")
for f in expected:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) // 1024 if exists else 0
    status = f'✅  {size:4d} KB' if exists else '❌  MISSING'
    print(f'{status}   {f}')


In [ ]:
# Colab direct download handler
try:
    from google.colab import files
    print("Downloading files directly to browser:")
    for f in expected:
        if os.path.exists(f):
            print(f'  Downloading: {f}')
            files.download(f)
except ImportError:
    print("Running locally. Models are located in your local project 'ml/runs/' folder.")